# 03 - Feature Engineering and Pre-Processing

##### 03-00 General Model Plan

Feature Engineering

- Create `duration_days` from the cleaned date columns
- Remove invalid and outlier duration rows
- Create the target variable from duration ranges
- Save the final cleaned modeling dataset

After feature engineering (Python training script): 

Pre-Processing

- Split dataset into training and testing subsets
- Split pre-processing steps into 3 groups (Text, Categories, and Words/Lengths)
- Use TF-IDF vectorization to convert text values (summaries and descriptions) to numeric matrices
- Use One Hot Encoder to encode catgories into numeric values with no relations, e.g, Bug and Improvement are completely unrelated
- Use Standard Scaler to standardizes the numeric values

Model training

- Use Logistic Regression
- Train and test model against test subset
- Evaluate and study model metrics


## 03-01 Importing CSV file and defining the target

Before we start training the model using the cleaned dataset, we need to create the target value from the cleaned date columns.

In [62]:
from pathlib import Path
import pandas as pd

jira_df = pd.read_csv("../data/processed/jira_issues_cleaned.csv", index_col=0)

task_df = jira_df.copy()

for col in ["created", "resolutiondate"]:
    task_df[col] = pd.to_datetime(task_df[col])

task_df.head()

,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Improvement,49,9,176,28,2005-01-01 07:47:50,2005-01-01 07:50:46
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,2004-12-25 22:50:30,2004-12-30 05:30:36
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,2004-12-27 01:34:17,2005-01-03 10:21:28
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,2004-12-28 18:50:52,2005-01-03 11:29:27
7,Port changes from tomcat-catalina,Since the [naming] sources were extracted from...,Major,Bug,33,4,1060,135,2005-01-03 11:16:07,2005-01-03 11:38:03


## 03-02 Create duration days derived variable

Calculate task duration in days.

This target is derived from:

`duration_days = resolutiondate - created`

In [63]:
task_df["duration_days"] = (task_df["resolutiondate"] - task_df["created"]).dt.total_seconds() / (60 * 60 * 24)

task_df["duration_days"].describe()

count    743631.000000
mean        123.703169
std         358.280600
min           0.000000
25%           0.963420
50%           8.163681
75%          61.880231
max        8001.517257
Name: duration_days, dtype: float64

The above statistics show target-distribution issues that need to be handled before modeling:

- The target has a strong long-tail pattern.
- Extreme outliers pull the mean far above the median.

#### Inspect long-tail durations

Before removing long-running tasks, first inspect how many rows would be affected.

In [64]:
display(task_df["duration_days"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

durations_over_120_days = (task_df["duration_days"] > 120).sum()
print(f"Rows with duration_days > 120: {durations_over_120_days:,}")

count    743631.000000
mean        123.703169
std         358.280600
min           0.000000
50%           8.163681
75%          61.880231
90%         321.861343
95%         699.164010
99%        1811.827084
max        8001.517257
Name: duration_days, dtype: float64

Rows with duration_days > 120: 134,559


Removing this many rows is a significant change, although the dataset may still contain enough records for modeling.

Tasks that exceed 90 days may be signs of:

- stale issues
- abandoned tickets
- imported/migrated issues
- tickets closed administratively
- issues left open for months/years without active work
- old backlog cleanup

These rows may not represent normal task-completion behavior, so removing or capping them can make the target easier for the model to learn.

In [65]:
task_df = task_df[task_df["duration_days"] <= 90].copy()

display(task_df.shape)
task_df.describe()

(587481, 11)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,587481.000000,587481.000000,5.874810e+05,587481.000000,587481,587481,587481.000000
mean,56.790557,7.854949,7.668200e+02,70.828474,2015-12-11 13:27:53.731751,2015-12-24 20:37:02.655466,13.298020
min,0.000000,0.000000,0.000000e+00,0.000000,2002-05-10 18:40:29,2002-05-10 18:54:17,0.000000
25%,40.000000,5.000000,1.060000e+02,15.000000,2012-09-07 22:54:13,2012-09-19 12:52:17,0.541134
50%,54.000000,7.000000,2.630000e+02,37.000000,2016-03-09 15:42:34,2016-03-22 21:43:55,3.978657
75%,70.000000,10.000000,6.100000e+02,78.000000,2019-10-04 21:09:22,2019-10-20 14:33:16,16.987106
max,255.000000,46.000000,1.602994e+06,139288.000000,2024-11-06 14:09:26,2025-01-27 21:10:15,89.999062
std,23.390909,3.578193,4.597455e+03,310.795385,NaN,NaN,19.961961


Tasks resolved in under 2 hours may represent very small fixes, administrative updates, or tickets closed immediately after creation. Removing them can reduce short-duration skew, but this should be treated as a modeling decision rather than a universal rule.

In [66]:
task_df = task_df[task_df["duration_days"] >= (1/24)].copy()

## 03-03 Create duration classification target

Create a classification target from `duration_days`.

The goal is to choose ranges that are:

- simple enough to explain
- not overly skewed toward one class
- aligned with what the task summaries and descriptions appear to represent

The earlier EDA showed two important cleanup decisions before classification:

- remove tasks completed in under 2 hours
- remove very long tasks above the selected long-tail cutoff

After those removals, compare a few possible class ranges before choosing the final classes.

### Duration ranges

- **Short:** same day and up to 3 days
- **Standard:** greater than 3 days and up to 3 weeks
- **Long-term:** greater than 3 weeks


In [67]:
def duration_category(days):
    if days <= 3:
        return "Short"
    if days <= 21:
        return "Standard"
    return "Long-running"

duration_order = ["Short", "Standard", "Long-running"]

task_df["duration_category"] = task_df["duration_days"].apply(duration_category)

class_counts = task_df["duration_category"].value_counts().reindex(duration_order)
class_percentages = (task_df["duration_category"].value_counts(normalize=True).reindex(duration_order).mul(100).round(2))

duration_class_summary = pd.DataFrame({
    "count": class_counts,
    "percent": class_percentages,
})

display(task_df.shape)
display(task_df.describe())

duration_class_summary

(518053, 12)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,518053.000000,518053.000000,5.180530e+05,518053.000000,518053,518053,518053.000000
mean,57.021662,7.870608,8.160859e+02,74.993804,2016-02-05 15:35:46.107054,2016-02-20 17:29:00.245780,15.078636
min,1.000000,1.000000,0.000000e+00,0.000000,2002-05-10 18:40:29,2002-06-03 18:35:07,0.041667
25%,41.000000,5.000000,1.170000e+02,16.000000,2012-11-14 18:29:15,2012-11-30 13:38:23,1.085995
50%,54.000000,7.000000,2.830000e+02,40.000000,2016-05-28 10:48:42,2016-06-11 01:40:29,5.723924
75%,70.000000,10.000000,6.540000e+02,82.000000,2019-12-18 08:14:16,2020-01-07 15:52:05,20.486921
max,254.000000,46.000000,1.602994e+06,139288.000000,2024-11-06 12:43:08,2025-01-27 21:10:15,89.999062
std,23.374343,3.578364,4.862103e+03,328.794957,NaN,NaN,20.616846


,count,percent
duration_category,,
Short,200277,38.66
Standard,191060,36.88
Long-running,126716,24.46


### 03-03.1 Spot-check task text by class

Sample a few summaries and descriptions from each final class. This is a manual sanity check that the classes are not obviously misleading for the type of task text the model will receive.

In [68]:
sample_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "duration_category",
]

for category in duration_order:
    print(category)
    class_sample = task_df.loc[
        task_df["duration_category"] == category,
        sample_columns,
    ].sample(n=5, random_state=42)
    display(class_sample)

Short


,summary,description,priority_name,issuetype_name,duration_category
280434,Return only local queue names for AddressContr...,There is a discrepancy between what {{AddressC...,Major,Bug,Short
113838,ComboBox should scroll the matched item to the...,Steps to reproduce: 1. Click in the ComboBox 2...,Major,Bug,Short
198984,Run slider-agent (py)tests from target directo...,I had to -1 the slider-0.90.2-incubating RC0 b...,Major,Improvement,Short
603779,Update 1.8.0 NOTICE.txt file to reflect 2017 c...,NaN,Major,Sub-task,Short
843797,[R] Turn linter off on curly braces on new line,Styler keeps introducing a change where a curl...,Major,Improvement,Short


Standard


,summary,description,priority_name,issuetype_name,duration_category
152922,TestAvroStorageUtils.testGetConcretePathFromGl...,I found that TestAvroStorageUtils.testGetConcr...,Major,Bug,Standard
152042,NPE during GC with HierarchicalLedgerManager,"{noformat} 2013-02-11 14:06:28,904 - WARN - [G...",Minor,Bug,Standard
285796,Enforce ErrorProne analysis in protobuf extens...,Java ErrorProne static analysis was [recently ...,P3,Improvement,Standard
336573,Add left over data check to generated C parsers,Fix generated C parsers not checking for left ...,Minor,Improvement,Standard
226991,Fixes examples of the official templates,First I will fix the following examples contai...,Minor,Sub-task,Standard


Long-running


,summary,description,priority_name,issuetype_name,duration_category
148496,Fix non-deterministic results in newline.q and...,newline.q and timestamp_lazy.q have non-determ...,Major,Bug,Long-running
963071,Sasl message with MD5 challenge text shouldn't...,Some log examples: {noformat} 2014-09-24 05:42...,Critical,Bug,Long-running
581017,docs - pxf install via command line is missing...,"add additional steps in ""Installing PXF Plug-i...",Major,Bug,Long-running
653558,insert data through jdbc while column type is ...,insert negative numeric (-1) through jdbc addb...,Major,Bug,Long-running
896946,Can't create system VM on XCP1.6 hypervisor,Hello I'm trying to set cloudstack v4 with XCP...,Critical,Bug,Long-running


The samples should not be expected to prove the classes are perfect. They only check that the ranges are reasonable enough for the first text-classification model. Actual performance should still be tested during model training.

## 03-04 Save final cleaned dataset

Save the final cleaned dataset after feature engineering.

We will also remove the `duration_days` at this stage to avoid leakage since the model is only going to be trained on the duration ranges for classification.

In [69]:
final_cleaned_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "duration_category",
]

final_cleaned_df = task_df[final_cleaned_columns].copy()
final_cleaned_df_sample = final_cleaned_df.sample(n=100, random_state=42)

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

final_cleaned_path = output_dir / "final_cleaned.csv"
final_cleaned_df.to_csv(final_cleaned_path, index=False)

final_cleaned_df_sample_path = output_dir / "final_cleaned_sample.csv"
final_cleaned_df_sample.to_csv(final_cleaned_df_sample_path, index=False)

print(f"Final cleaned modeling rows: {final_cleaned_df.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df.shape[1]:,}")
print(f"Saved final cleaned CSV file to: {final_cleaned_path}")
print(f"Saved final cleaned sample CSV file to: {final_cleaned_df_sample_path}")

final_cleaned_df.head()

Final cleaned modeling rows: 518,053
Final cleaned modeling columns: 9
Saved final cleaned CSV file to: ..\data\processed\final_cleaned.csv
Saved final cleaned sample CSV file to: ..\data\processed\final_cleaned_sample.csv


,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,duration_category
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,Standard
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,Standard
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,Standard
8,Groovy / Java Mismatch with simple class,The following code compiles and runs as a java...,Major,Bug,40,7,1438,65,Long-running
9,ActionContext should always have a TextProvide...,Right now if you access the ActionContext befo...,Minor,Improvement,68,10,210,33,Short


In [61]:
print(f"Final cleaned modeling rows: {final_cleaned_df_sample.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df_sample.shape[1]:,}")

Final cleaned modeling rows: 100
Final cleaned modeling columns: 9
